# Классификация поворота текста: 0° / 180°

**Решение** — смесь двух маленьких моделей каждая с TTA (test-time augmentation)
поворотом на 180°: `p_180 = 0.2 · MobileNetV3-Small + 0.8 · PP-LCNet_x1_0_textline_ori`.

## Подход

1. **Данные.** Обучающих данных в задаче нет, поэтому для дообучения собран датасет на 500k изображений:
   350k реальных слов из [TextOCR](https://textvqa.org/textocr/) и 150k синтетических строк (≈50% на русском языке).
   Класс 1 — то же изображение, повёрнутое на 180°. Разбиение train/val/test 70/15/15.
2. **Модели.** MobileNetV3-Small (ImageNet-инициализация), дообученная на этом датасете: вход 48×320,
   выбор эпохи по Brier на OCR-части val (`training/train_mobilenet.ipynb`).
   PP-LCNet_x1_0_textline_ori — готовый классификатор ориентации строки из PaddleOCR, **без дообучения**.
3. **TTA.** Каждая модель предсказывает и по изображению, и по его повороту на 180°, а ответы объединяются
   с учётом того, что поворот меняет метку, — это убирает смещение модели к одному из классов.
   TTA дал основной прирост: +0.016 у MobileNet, +0.045 у PP-LCNet x0.25.
4. **Смесь.** Ошибки моделей слабо коррелируют (они обучены на разных данных), поэтому смесь с весами 0.2 / 0.8
   лучше каждой из моделей.

| сабмит | 1 − Brier |
|---|---|
| MobileNetV3-Small, дообученная, без TTA | 0.8922 |
| MobileNetV3-Small + TTA | 0.9081 |
| PP-LCNet x1.0 + TTA | 0.9595 |
| **смесь 0.2 / 0.8** | **0.9636** |

In [1]:
# Импорты и настройки параметров
import os
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'  # веса локальные
from paddleocr import TextLineOrientationClassification

TEST_DIR = Path('test/images')
WEIGHTS_DIR = Path('weights')
OUTPUT = Path('submission.csv')

W_MOBILENET, W_PADDLE = 0.2, 0.8  # веса моделей в финальном решении
CHUNK = 2048                      # изображения читаются частями, чтобы не держать весь объем в памяти

/home/user_f/projects/avito-bootcamp-2026-autumn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Модель 1: MobileNetV3-Small, дообученная, в формате ONNX.
# Препроцессинг: изображение вписывается в 48×320 с сохранением пропорций,
# поля серые и симметричные, нормализация ImageNet. Модель выдаёт уверенность в повороте на 180°.
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)  # средние и стандартные отклонения ImageNet (RGB),
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)   

# ONNX Runtime на CPU: так инференсу не нужен torch (его CUDA-библиотеки конфликтуют с paddlepaddle-gpu)
mobilenet = ort.InferenceSession(str(WEIGHTS_DIR / 'mobilenet_v3_small_orientation.onnx'),
                                 providers=['CPUExecutionProvider'])


def sigmoid(z):
    """Переводит логиты в вероятности."""
    return 1 / (1 + np.exp(-np.clip(z, -80, 80)))


def letterbox(image_bgr, height=48, width=320):
    """Готовит изображение для MobileNet точно так же, как при обучении.

    Вписывает его в height×width без искажения пропорций и добавляет серые поля поровну с двух сторон.
    Возвращает нормализованный массив 3×height×width.
    """
    image = Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    scale = min(width / image.width, height / image.height)  # масштаб, при котором изображение целиком помещается
    w, h = max(1, round(image.width * scale)), max(1, round(image.height * scale))
    canvas = Image.new('RGB', (width, height), (127, 127, 127))
    canvas.paste(image.resize((w, h), Image.Resampling.BILINEAR), ((width - w) // 2, (height - h) // 2))  # по центру
    x = (np.asarray(canvas, dtype=np.float32) / 255 - MEAN) / STD
    return x.transpose(2, 0, 1)  # H×W×C -> C×H×W


def mobilenet_p180(images, batch_size=256):
    """Вероятность поворота на 180° по MobileNet для списка изображений (BGR)."""
    probs = []
    for i in range(0, len(images), batch_size):
        x = np.stack([letterbox(image) for image in images[i:i + batch_size]]) # np.stack для ONNX совместимости
        logits = mobilenet.run(None, {'image': x})[0].ravel()  # выход (batch, 1) -> (batch,)
        probs.append(sigmoid(logits))
    return np.concatenate(probs)

In [ ]:
# Модель 2: PP-LCNet_x1_0_textline_ori из PaddleOCR, без дообучения.
# Классы — 0_degree и 180_degree.
# model_dir — веса из репозитория, без скачивания
paddle = TextLineOrientationClassification(model_name='PP-LCNet_x1_0_textline_ori',
                                           model_dir=str(WEIGHTS_DIR / 'PP-LCNet_x1_0_textline_ori'))


def paddle_p180(images, batch_size=256):
    """Вероятность поворота на 180° по PP-LCNet для списка изображений (BGR).

    Препроцессинг модель делает сама. Для каждого изображения она возвращает названия классов и их скоры.
    """
    probs = []
    for result in paddle.predict(input=images, batch_size=batch_size):
        scores = dict(zip(result['label_names'], result['scores']))
        # обычно есть скор класса 180_degree; если модель вернула только 0_degree, то берём дополнение до 1
        probs.append(scores['180_degree'] if '180_degree' in scores else 1 - scores['0_degree'])
    return np.array(probs)

/home/user_f/projects/avito-bootcamp-2026-autumn/.venv/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [ ]:
# TTA: поворот на 180°.
# Модель смотрит на изображение и на его поворот. Поворот меняет правильный ответ на противоположный
def logit(p, eps=1e-6):
    """Обратная к sigmoid функция: вероятность -> логит."""
    p = np.clip(p, eps, 1 - eps)  # при p = 0 или 1 логит бесконечен
    return np.log(p / (1 - p))


def predict_with_tta(predict, images):
    """Вероятность поворота на 180° с учётом TTA.

    predict — функция модели (mobilenet_p180 или paddle_p180). Считает её ответ на исходных изображениях
    и на повёрнутых на 180°. Для повёрнутого изображения правильный ответ противоположный, поэтому его логит
    берётся со знаком минус, и два ответа усредняются: sigmoid((logit p(x) − logit p(rot180(x))) / 2).
    Если модель в целом склонна к одному из классов, это смещение одинаково входит в оба логита и сокращается.
    """
    # float64: у вероятностей около 1 точности float32 не хватает для логита
    p = predict(images).astype(np.float64)
    # [::-1, ::-1] переворачивает строки и столбцы, то есть поворачивает на 180°;
    # ascontiguousarray нужен из-за того, что OpenCV и Paddle не работают с отрицательными шагами массива
    p_rotated = predict([np.ascontiguousarray(image[::-1, ::-1]) for image in images]).astype(np.float64)
    return sigmoid((logit(p) - logit(p_rotated)) / 2)

In [ ]:
# Предсказание: обе модели с TTA, смесь 0.2 / 0.8, запись submission.csv
paths = sorted(TEST_DIR.glob('*.png'))  # сортировка по имени
print(f'{len(paths)} изображений в {TEST_DIR}')

p_180 = []
for start in tqdm(range(0, len(paths), CHUNK)):
    images = [cv2.imread(str(path)) for path in paths[start:start + CHUNK]]
    p_mobilenet = predict_with_tta(mobilenet_p180, images)
    p_paddle = predict_with_tta(paddle_p180, images)
    p_180.append(W_MOBILENET * p_mobilenet + W_PADDLE * p_paddle) # результат — смесь двух моделей

submission = pd.DataFrame({'image_id': [path.stem for path in paths], 'p_180': np.concatenate(p_180)})
submission.to_csv(OUTPUT, index=False, float_format='%.5f')  # до 5 знаков после запятой
print(f'Сохранено: {OUTPUT}, {len(submission)} строк')

20000 изображений в test/images



  0%|          | 0/10 [00:00<?, ?it/s]


 10%|█         | 1/10 [00:24<03:39, 24.34s/it]


 20%|██        | 2/10 [00:44<02:55, 21.92s/it]


 30%|███       | 3/10 [00:58<02:08, 18.43s/it]

...


 80%|████████  | 8/10 [02:17<00:31, 15.60s/it]


 90%|█████████ | 9/10 [02:35<00:16, 16.18s/it]


100%|██████████| 10/10 [02:52<00:00, 16.53s/it]


100%|██████████| 10/10 [02:52<00:00, 17.26s/it]

Сохранено: submission.csv, 20000 строк
